# Geração de Dados Fictícios — E-commerce

Neste notebook geramos dados fictícios de um e-commerce utilizando a biblioteca Faker.
São criados 4 arquivos CSV que simulam clientes, produtos, pedidos e itens de pedido de 2 anos de operação.

## 1. Importações e Configurações

In [1]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta
import os

fake = Faker("pt_BR")
random.seed(42)
Faker.seed(42)

NUM_CLIENTES = 500
NUM_PRODUTOS = 80
NUM_PEDIDOS  = 3000

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configurações prontas!")

Configurações prontas!


## 2. Dados de Referência

Definimos as categorias de produtos com faixas de preço, os possíveis status de pedido com seus pesos de probabilidade e as formas de pagamento.

In [2]:
CATEGORIAS = {
    "Eletrônicos":    (150.0,  4500.0),
    "Vestuário":      (29.90,  399.90),
    "Casa & Jardim":  (19.90,  899.90),
    "Esportes":       (39.90,  1299.90),
    "Beleza":         (12.90,  299.90),
    "Livros":         (19.90,  149.90),
    "Brinquedos":     (24.90,  599.90),
    "Alimentos":      (8.90,   129.90),
}

STATUS_PEDIDO = ["concluído", "enviado", "processando", "cancelado"]
STATUS_PESO   = [0.60,        0.20,      0.12,          0.08]

FORMAS_PAGAMENTO = ["cartão crédito", "cartão débito", "pix", "boleto"]
PAGAMENTO_PESO   = [0.45,             0.20,            0.25,  0.10]

REGIOES = {
    "SP": "Sudeste", "RJ": "Sudeste", "MG": "Sudeste", "ES": "Sudeste",
    "RS": "Sul",     "SC": "Sul",     "PR": "Sul",
    "BA": "Nordeste","PE": "Nordeste","CE": "Nordeste","MA": "Nordeste",
    "AM": "Norte",   "PA": "Norte",   "RO": "Norte",
    "GO": "Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste","DF":"Centro-Oeste",
}

print("Dados de referência carregados!")

Dados de referência carregados!


## 3. Gerando Clientes

Criamos 500 clientes fictícios com dados pessoais e localização. O estado é escolhido aleatoriamente e a região é mapeada automaticamente.

In [3]:
def gerar_clientes(n):
    registros = []
    for i in range(1, n + 1):
        estado = random.choice(list(REGIOES.keys()))
        registros.append({
            "cliente_id":    i,
            "nome":          fake.name(),
            "email":         fake.email(),
            "telefone":      fake.phone_number(),
            "cidade":        fake.city(),
            "estado":        estado,
            "regiao":        REGIOES[estado],
            "data_cadastro": fake.date_between(start_date="-4y", end_date="today").strftime("%Y-%m-%d"),
            "genero":        random.choice(["M", "F", "Outro"]),
            "idade":         random.randint(18, 75),
        })
    return pd.DataFrame(registros)

clientes = gerar_clientes(NUM_CLIENTES)
print(f"{len(clientes)} clientes gerados!")
clientes.head()

500 clientes gerados!


,cliente_id,nome,email,telefone,cidade,estado,regiao,data_cadastro,genero,idade
0,1,Brenda Alves,samuel32@example.net,+55 51 8196 0013,Montenegro,ES,Sudeste,2024-12-24,M,65
1,2,Gael Henrique Silva,ccamara@example.net,+55 84 0265-4235,Borges,PE,Nordeste,2024-01-14,M,32
2,3,Luiz Henrique Ferreira,caio78@example.org,41 1849 5931,Ribeiro,RS,Sul,2023-05-23,Outro,24
3,4,Bruno da Conceição,garciaagatha@example.com,21 2553 4192,Castro,DF,Centro-Oeste,2024-05-21,M,55
4,5,Júlia Porto,ayllavargas@example.org,+55 31 0564-1395,Rezende,RO,Norte,2024-07-17,M,19


## 4. Gerando Produtos

Criamos 80 produtos com preço de custo e venda calculados por categoria. A margem é calculada automaticamente.

In [4]:
NOMES_PRODUTO = [
    "Pro", "Ultra", "Max", "Elite", "Basic", "Smart", "Eco", "Plus",
    "Mini", "Grande", "Premium", "Lite", "Turbo", "Neo", "Air",
]

def gerar_nome_produto(categoria):
    adjetivo = random.choice(NOMES_PRODUTO)
    sufixo   = fake.word().capitalize()
    return f"{adjetivo} {sufixo} {categoria.split()[0]}"

def gerar_produtos(n):
    registros = []
    for i in range(1, n + 1):
        categoria            = random.choice(list(CATEGORIAS.keys()))
        preco_min, preco_max = CATEGORIAS[categoria]
        preco_custo          = round(random.uniform(preco_min * 0.4, preco_max * 0.5), 2)
        preco_venda          = round(preco_custo * random.uniform(1.4, 2.8), 2)
        registros.append({
            "produto_id":  i,
            "nome":        gerar_nome_produto(categoria),
            "categoria":   categoria,
            "preco_custo": preco_custo,
            "preco_venda": preco_venda,
            "margem_pct":  round((preco_venda - preco_custo) / preco_venda * 100, 1),
            "estoque":     random.randint(0, 500),
            "ativo":       random.choices([True, False], weights=[0.90, 0.10])[0],
            "fornecedor":  fake.company(),
        })
    return pd.DataFrame(registros)

produtos = gerar_produtos(NUM_PRODUTOS)
print(f"{len(produtos)} produtos gerados!")
produtos.head()

80 produtos gerados!


,produto_id,nome,categoria,preco_custo,preco_venda,margem_pct,estoque,ativo,fornecedor
0,1,Lite Sit Beleza,Beleza,95.55,212.07,54.9,166,True,Marques
1,2,Ultra Molestiae Beleza,Beleza,70.85,159.24,55.5,406,True,Sampaio Nunes S/A
2,3,Eco Mollitia Esportes,Esportes,288.62,762.92,62.2,173,True,da Costa S.A.
3,4,Eco Harum Brinquedos,Brinquedos,130.59,200.21,34.8,160,True,Rios
4,5,Plus Atque Livros,Livros,71.84,169.65,57.7,34,True,Monteiro Jesus - EI


## 5. Gerando Pedidos e Itens

Criamos 3.000 pedidos distribuídos ao longo de 2 anos. Cada pedido tem entre 1 e 5 itens escolhidos aleatoriamente do catálogo de produtos.

In [5]:
def gerar_pedidos(n, clientes):
    data_inicio = datetime.now() - timedelta(days=365 * 2)
    registros   = []
    for i in range(1, n + 1):
        data_pedido = fake.date_time_between(start_date=data_inicio, end_date="now")
        status      = random.choices(STATUS_PEDIDO, STATUS_PESO)[0]
        data_entrega = (data_pedido + timedelta(days=random.randint(1, 15))).strftime("%Y-%m-%d") if status == "concluído" else None
        registros.append({
            "pedido_id":       i,
            "cliente_id":      random.choice(clientes["cliente_id"].tolist()),
            "data_pedido":     data_pedido.strftime("%Y-%m-%d"),
            "hora_pedido":     data_pedido.strftime("%H:%M:%S"),
            "status":          status,
            "forma_pagamento": random.choices(FORMAS_PAGAMENTO, PAGAMENTO_PESO)[0],
            "frete":           round(random.uniform(0, 49.90), 2),
            "desconto_pct":    random.choices([0, 5, 10, 15, 20], weights=[0.5, 0.2, 0.15, 0.1, 0.05])[0],
            "data_entrega":    data_entrega,
            "canal_venda":     random.choice(["site", "app", "marketplace", "loja física"]),
        })
    return pd.DataFrame(registros)

def gerar_itens(pedidos, produtos):
    registros = []
    item_id   = 1
    for _, pedido in pedidos.iterrows():
        n_itens = random.choices([1, 2, 3, 4, 5], weights=[0.45, 0.30, 0.15, 0.07, 0.03])[0]
        for _, prod in produtos.sample(n_itens).iterrows():
            quantidade = random.randint(1, 5)
            subtotal   = round(quantidade * prod["preco_venda"], 2)
            registros.append({
                "item_id":    item_id,
                "pedido_id":  pedido["pedido_id"],
                "produto_id": prod["produto_id"],
                "quantidade": quantidade,
                "preco_unit": prod["preco_venda"],
                "subtotal":   subtotal,
            })
            item_id += 1
    return pd.DataFrame(registros)

pedidos = gerar_pedidos(NUM_PEDIDOS, clientes)
itens   = gerar_itens(pedidos, produtos)
print(f"{len(pedidos)} pedidos gerados!")
print(f"{len(itens)} itens gerados!")
pedidos.head()

3000 pedidos gerados!
5719 itens gerados!


,pedido_id,cliente_id,data_pedido,hora_pedido,status,forma_pagamento,frete,desconto_pct,data_entrega,canal_venda
0,1,266,2024-10-02,02:04:15,concluído,cartão crédito,4.12,0,2024-10-15,loja física
1,2,422,2026-05-16,11:44:06,enviado,cartão crédito,14.66,0,NaN,marketplace
2,3,157,2024-10-29,04:40:46,enviado,cartão crédito,35.81,0,NaN,loja física
3,4,373,2024-06-07,03:43:45,concluído,pix,17.59,20,2024-06-08,marketplace
4,5,406,2024-06-13,21:02:46,enviado,boleto,3.74,5,NaN,marketplace


## 6. Salvando os CSVs

In [6]:
def adicionar_totais(pedidos, itens):
    totais = itens.groupby("pedido_id")["subtotal"].sum().reset_index()
    totais.rename(columns={"subtotal": "subtotal_produtos"}, inplace=True)
    pedidos = pedidos.merge(totais, on="pedido_id", how="left")
    pedidos["desconto_valor"] = round(pedidos["subtotal_produtos"] * pedidos["desconto_pct"] / 100, 2)
    pedidos["valor_total"]    = round(pedidos["subtotal_produtos"] - pedidos["desconto_valor"] + pedidos["frete"], 2)
    return pedidos

pedidos = adicionar_totais(pedidos, itens)

clientes.to_csv(f"{OUTPUT_DIR}/clientes.csv",      index=False, encoding="utf-8")
produtos.to_csv(f"{OUTPUT_DIR}/produtos.csv",      index=False, encoding="utf-8")
pedidos.to_csv( f"{OUTPUT_DIR}/pedidos.csv",       index=False, encoding="utf-8")
itens.to_csv(   f"{OUTPUT_DIR}/itens_pedido.csv",  index=False, encoding="utf-8")

print("✓ CSVs salvos em data/raw/")
print(f"  Faturamento total: R$ {pedidos['valor_total'].sum():,.2f}")
print(f"  Ticket médio:      R$ {pedidos['valor_total'].mean():,.2f}")

✓ CSVs salvos em data/raw/
  Faturamento total: R$ 8,125,151.97
  Ticket médio:      R$ 2,708.38
